# Remapping Warna Sera-Z

Diperbarui: 25 November 2026
PIC: Bisma Adhiaksa

Notebook ini menjadi acuan Izza untuk pekerjaan pencocokan warna. Isinya:

1. Tiga kosakata warna yang dipakai proyek.
2. Aturan pemetaan 16 keluarga internal ke 14 palet UI dan 14 warna tabel ML.
3. Kode pemetaan dan validasi.
4. Hasil pengecekan terhadap database `assets/db/fashion_app.db`.

## 1. Tiga kosakata warna

| Kosakata | Jumlah | Dipakai oleh | Contoh |
|---|---:|---|---|
| `ColorFamily` internal | 16 | harmoni, edit warna, UI | `cream`, `maroon`, `olive`, `teal`, `navy` |
| Palet tampilan | 14 | teks UI bahasa Indonesia | `krem`, `merah`, `hijau army`, `biru muda` |
| Tabel ML/SQLite | 14 | `ml_pair_score`, `ml_style`, `ml_formality` | `beige`, `red`, `green`, `blue` |

Warna hasil ekstraksi atau pilihan pengguna boleh memakai 16 keluarga. Sebelum
dikirim ke tabel ML, warna harus dipetakan ke 14 nama database. Tanpa
pemetaan ini, warna seperti `cream`, `maroon`, `olive`, dan `teal` tidak akan
menemukan skor pasangan.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

PALETTE_14 = [
    "hitam", "putih", "abu-abu", "krem", "cokelat",
    "merah", "pink", "oranye", "kuning", "hijau",
    "hijau army", "biru muda", "navy", "ungu",
]

ML_FAMILY_14 = [
    "beige", "black", "blue", "brown", "gray", "green",
    "multicolor", "orange", "other", "pink", "purple",
    "red", "white", "yellow",
]

# canonical, label, hex, palet UI, warna tabel ML
FAMILY_16 = [
    ("black", "Hitam", "#1A1A1A", "hitam", "black"),
    ("white", "Putih", "#F5F5F5", "putih", "white"),
    ("gray", "Abu-abu", "#808080", "abu-abu", "gray"),
    ("cream", "Krem", "#F2E3C6", "krem", "beige"),
    ("brown", "Cokelat", "#6B4423", "cokelat", "brown"),
    ("red", "Merah", "#C62828", "merah", "red"),
    ("maroon", "Maroon", "#7B1F2B", "merah", "red"),
    ("pink", "Pink", "#E8A0BF", "pink", "pink"),
    ("orange", "Oranye", "#E67E22", "oranye", "orange"),
    ("yellow", "Kuning", "#E6C229", "kuning", "yellow"),
    ("green", "Hijau", "#3E7C4F", "hijau", "green"),
    ("olive", "Olive", "#6B8E23", "hijau army", "green"),
    ("teal", "Teal", "#00897B", "biru muda", "blue"),
    ("blue", "Biru", "#2F5D9E", "biru muda", "blue"),
    ("navy", "Navy", "#1A237E", "navy", "blue"),
    ("purple", "Ungu", "#6A1B9A", "ungu", "purple"),
]

print("16 keluarga:", len(FAMILY_16))
print("14 palet UI:", len(PALETTE_14))
print("14 warna ML:", len(ML_FAMILY_14))

16 keluarga: 16
14 palet UI: 14
14 warna ML: 14


## 2. Aturan pemetaan

Alias berikut diterima dari data lama, label UI, atau dataset:

| Input | Canonical |
|---|---|
| `beige` | `cream` |
| `grey` | `gray` |
| `sage`, `army green`, `hijau army` | `olive` |
| `denim`, `light blue`, `biru muda` | `blue` |

Keputusan warna yang tidak punya nama sendiri di palet 14:

| Canonical | Palet UI | Tabel ML | Alasan |
|---|---|---|---|
| `cream` | `krem` | `beige` | nama lama database |
| `maroon` | `merah` | `red` | keluarga merah gelap |
| `olive` | `hijau army` | `green` | keluarga hijau |
| `teal` | `biru muda` | `blue` | keluarga biru-hijau |
| `navy` | `navy` | `blue` | tabel ML belum memiliki navy |

Pemetaan ini eksplisit. Nilai yang tidak dikenal dipetakan ke `other` untuk
database, bukan dibiarkan gagal mencari skor.

In [2]:
ALIASES = {
    "grey": "gray",
    "beige": "cream",
    "sage": "olive",
    "army green": "olive",
    "hijau army": "olive",
    "denim": "blue",
    "light blue": "blue",
    "hitam": "black",
    "putih": "white",
    "abu-abu": "gray",
    "krem": "cream",
    "cokelat": "brown",
    "merah": "red",
    "merah muda": "pink",
    "oranye": "orange",
    "kuning": "yellow",
    "hijau": "green",
    "biru muda": "blue",
    "ungu": "purple",
}

DB_BY_CANONICAL = {
    row[0]: row[4] for row in FAMILY_16
}
DB_BY_CANONICAL.update({"other": "other", "multicolor": "multicolor"})


def canonical_key(family):
    """Normalisasi input menjadi kunci 16 keluarga internal."""
    raw = str(family or "other").strip().lower()
    if not raw:
        return "other"
    return ALIASES.get(raw, raw)


def database_family(family):
    """Nama warna untuk tabel ML/SQLite 14 warna."""
    return DB_BY_CANONICAL.get(canonical_key(family), "other")


def palette_name(family):
    """Nama warna untuk teks UI 14 palet."""
    canonical = canonical_key(family)
    for key, _, _, palette, _ in FAMILY_16:
        if key == canonical:
            return palette
    return "other"


def sql_case(column):
    """Ekspresi SQL untuk memetakan kolom 16 warna ke 14 warna ML."""
    cases = []
    for key, db_family in DB_BY_CANONICAL.items():
        if key != db_family:
            cases.append(f"WHEN {column} = '{key}' THEN '{db_family}'")
    return "CASE " + column + " " + " ".join(cases) + " ELSE " + column + " END"


print("canonical beige ->", canonical_key("beige"))
print("database maroon ->", database_family("maroon"))
print("palette teal ->", palette_name("teal"))

canonical beige -> cream
database maroon -> red
palette teal -> biru muda


## 3. Tabel pemetaan lengkap

Tabel berikut menjadi acuan saat membaca hasil ekstraksi warna atau mengubah
warna pakaian di aplikasi.

In [3]:
mapping = pd.DataFrame(FAMILY_16, columns=[
    "canonical", "label", "hex", "palet_ui", "warna_ml",
])
mapping

,canonical,label,hex,palet_ui,warna_ml
0,black,Hitam,#1A1A1A,hitam,black
1,white,Putih,#F5F5F5,putih,white
2,gray,Abu-abu,#808080,abu-abu,gray
3,cream,Krem,#F2E3C6,krem,beige
4,brown,Cokelat,#6B4423,cokelat,brown
5,red,Merah,#C62828,merah,red
6,maroon,Maroon,#7B1F2B,merah,red
7,pink,Pink,#E8A0BF,pink,pink
8,orange,Oranye,#E67E22,oranye,orange
9,yellow,Kuning,#E6C229,kuning,yellow


## 4. Validasi pemetaan

Validasi memastikan seluruh 16 keluarga memiliki tujuan yang sah dan tidak
ada warna yang tertinggal ke `other` tanpa keputusan.

In [4]:
assert len(FAMILY_16) == 16
assert len({row[0] for row in FAMILY_16}) == 16
assert len(PALETTE_14) == 14
assert len(ML_FAMILY_14) == 14

for key, _, _, palette, db_family in FAMILY_16:
    assert palette in PALETTE_14, (key, palette)
    assert db_family in ML_FAMILY_14, (key, db_family)
    assert database_family(key) == db_family, key

for palette in PALETTE_14:
    assert palette_name(palette) == palette, palette

assert database_family("tidak_ada") == "other"
assert canonical_key("BIRU MUDA") == "blue"
print("OK: 16 keluarga, 14 palet UI, dan 14 warna ML konsisten")

OK: 16 keluarga, 14 palet UI, dan 14 warna ML konsisten


## 5. Hasil pengecekan database

Database asset saat ini masih memakai 14 warna. Pengecekan memastikan semua
warna tabel ML tercakup dan semua 16 keluarga menghasilkan warna yang dapat
dicari di `ml_pair_score`.

Catatan: `navy` ada di palet UI, tetapi tidak ada di `ml_pair_score`. Untuk
tabel ML, `navy` dipetakan ke `blue`.

In [5]:
ROOT = Path.cwd()
while not (ROOT / "assets" / "db" / "fashion_app.db").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DB_PATH = ROOT / "assets" / "db" / "fashion_app.db"
print("Database:", DB_PATH)

conn = sqlite3.connect(DB_PATH)
try:
    db_colors = {row[0] for row in conn.execute(
        "SELECT DISTINCT color_a FROM ml_pair_score"
    )}
    total_pairs = conn.execute("SELECT COUNT(*) FROM ml_pair_score").fetchone()[0]
finally:
    conn.close()

mapped_colors = {database_family(row[0]) for row in FAMILY_16}
missing = sorted(mapped_colors - db_colors)

print("Jumlah baris ml_pair_score:", total_pairs)
print("Warna database:", sorted(db_colors))
print("Warna hasil mapping:", sorted(mapped_colors))
print("Warna mapping yang tidak ada di database:", missing)

Database: D:\Kuliah\Semester5\PBL\assets\db\fashion_app.db
Jumlah baris ml_pair_score: 51352
Warna database: ['beige', 'black', 'blue', 'brown', 'gray', 'green', 'multicolor', 'orange', 'other', 'pink', 'purple', 'red', 'white', 'yellow']
Warna hasil mapping: ['beige', 'black', 'blue', 'brown', 'gray', 'green', 'orange', 'pink', 'purple', 'red', 'white', 'yellow']
Warna mapping yang tidak ada di database: []


## 6. Contoh input lama dan input UI

Tabel ini dipakai saat notebook pencocokan warna Izza menghasilkan nama lama,
nama Indonesia, atau nama 16 keluarga.

In [6]:
examples = [
    "beige", "grey", "sage", "denim",
    "hijau army", "BIRU MUDA", "maroon", "teal",
    "navy", "tidak_ada", "other",
]
pd.DataFrame([
    {
        "input": value,
        "canonical": canonical_key(value),
        "palet_ui": palette_name(value),
        "warna_ml": database_family(value),
    }
    for value in examples
])

,input,canonical,palet_ui,warna_ml
0,beige,cream,krem,beige
1,grey,gray,abu-abu,gray
2,sage,olive,hijau army,green
3,denim,blue,biru muda,blue
4,hijau army,olive,hijau army,green
5,BIRU MUDA,blue,biru muda,blue
6,maroon,maroon,merah,red
7,teal,teal,biru muda,blue
8,navy,navy,navy,blue
9,tidak_ada,tidak_ada,other,other


## 7. Catatan untuk Izza

- Gunakan 16 keluarga sebagai nama internal hasil pencocokan warna.
- Sebelum mencari skor pasangan di tabel ML, panggil `database_family()`.
- Jangan menulis `cream`, `maroon`, `olive`, atau `teal` langsung ke query
  `ml_pair_score`; tabel itu masih memakai 14 nama.
- Gunakan `palette_name()` hanya untuk teks UI.
- Semua warna yang tidak dikenal harus menjadi `other` untuk database.
- Bandingkan hasil notebookmu dengan tabel pada bagian 3 sebelum integrasi.

## 8. Ringkasan keputusan

| Warna internal | Palet UI | Database ML |
|---|---|---|
| `cream` | `krem` | `beige` |
| `maroon` | `merah` | `red` |
| `olive` | `hijau army` | `green` |
| `teal` | `biru muda` | `blue` |
| `navy` | `navy` | `blue` |

Pemetaan ini menjaga kompatibilitas dengan tabel ML yang sudah ada tanpa
mengubah skema database.